In [1]:
# Installation de la bibliothèque 'diffusers' pour le VAE de Stable Diffusion
!pip install diffusers transformers accelerate

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\benny\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from diffusers import AutoencoderKL

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation de : {device}")

# Normalisation standard ImageNet pour ResNet
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

C:\Users\benny\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Utilisation de : cpu


In [3]:
classifier = models.resnet50(pretrained=True).to(device)
classifier.eval()

# 2. Le VAE (celui qui remplit les zones vides de façon réaliste)
# 'sd-vae-ft-mse' est connu pour sa grande fidélité de reconstruction
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
vae.eval()

# On gèle les paramètres : on n'entraîne pas les modèles, on optimise seulement le masque
for param in list(classifier.parameters()) + list(vae.parameters()):
    param.requires_grad = False

C:\Users\benny\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\benny\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\benny\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:143: Us

In [ ]:
class VAEInfiller(nn.Module):
    def __init__(self, vae_model):
        super().__init__()
        self.vae = vae_model

    def forward(self, x_resnet, mask):
        # 1. ResNet [Norm] -> Image [0, 1]
        x_01 = x_resnet * STD + MEAN
        
        # 2. Image [0, 1] -> VAE [-1, 1]
        x_vae = x_01 * 2.0 - 1.0
        
        # 3. Masquage avec bruit (pour forcer le VAE à inventer du contenu)
        noise = torch.randn_like(x_vae)
        masked_input = x_vae * mask + noise * (1 - mask)
        
        # 4. Reconstruction par le VAE
        with torch.no_grad():
            latents = self.vae.encode(masked_input).latent_dist.sample()
            reconstruction = self.vae.decode(latents).sample
            
        # 5. Retour au format ResNet
        reconstruction_01 = (reconstruction + 1.0) / 2.0
        reconstruction_res = (reconstruction_01 - MEAN) / STD
        
        # On remplace les zones mask=0 par la reconstruction
        return x_resnet * mask + reconstruction_res * (1 - mask)

In [ ]:
def optimize_fido_mask(img_tensor, target_class, infiller, mode='SSR', steps=100):
    # On crée un petit masque (56x56) pour qu'il soit "lisse" une fois agrandi
    mask_logits = torch.ones(1, 1, 56, 56, device=device, requires_grad=True)
    optimizer = torch.optim.Adam([mask_logits], lr=0.1)
    
    for i in range(steps):
        optimizer.zero_grad()
        
        # Gumbel-Softmax : rend le choix binaire (0 ou 1) dérivable pour l'IA
        # On simule un masque binaire
        probs = torch.sigmoid(mask_logits)
        mask = F.interpolate(probs, size=(224, 224), mode='bilinear')
        
        # On crée l'image contrefactuelle
        infilled_img = infiller(img_tensor, mask)
        
        # On regarde le score du classifieur
        output = classifier(infilled_img)
        score = output[0, target_class]
        
        # Calcul de la perte (Loss)
        l1_loss = torch.mean(mask) # Taille du masque
        if mode == 'SSR':
            # Garder la classe avec le moins de pixels possible
            loss = -score + 0.5 * l1_loss 
        else:
            # Faire tomber la classe en masquant le moins de pixels possible
            loss = score + 0.5 * (1 - l1_loss)
            
        loss.backward()
        optimizer.step()
        
    return torch.sigmoid(mask_logits).detach()

In [ ]:
# Charger une image (remplacez par votre lien ou chemin)
# !wget https://image.url -O test.jpg
img_path = "votre_image.jpg" 
raw_img = Image.open(img_path).convert('RGB').resize((224, 224))
img_tensor = (transforms.ToTensor()(raw_img).unsqueeze(0).to(device) - MEAN) / STD

# 1. Trouver la classe
target = classifier(img_tensor).argmax().item()

# 2. Lancer FIDO
infiller = VAEInfiller(vae)
mask_result = optimize_fido_mask(img_tensor, target, infiller, mode='SSR')

# 3. Afficher
plt.imshow(raw_img)
mask_np = F.interpolate(mask_result, size=(224, 224)).cpu().squeeze().numpy()
plt.imshow(mask_np, cmap='jet', alpha=0.5)
plt.title(f"Zone critique pour la classe {target}")
plt.show()